In [1]:
# Function 3 - BBO Capstone Project
# Strategy: Gaussian Process + EI/UCB Hybrid
# Function 3 is a 3D black-box optimization problem.

import numpy as np
from scipy.stats import norm, qmc
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel


# --------------------------------------------------
# 1. Function 3 data
# --------------------------------------------------

X = np.array([
    [0.17152521, 0.34391687, 0.24873720],
    [0.24211446, 0.64407427, 0.27243281],
    [0.53490572, 0.39850092, 0.17338873],
    [0.49258141, 0.61159319, 0.34017639],
    [0.13462167, 0.21991724, 0.45820622],
    [0.34552327, 0.94135983, 0.26936348],
    [0.15183663, 0.43999062, 0.99088187],
    [0.64550284, 0.39714294, 0.91977134],
    [0.74691195, 0.28419631, 0.22629985],
    [0.17047699, 0.69703240, 0.14916943],
    [0.22054934, 0.29782524, 0.34355534],
    [0.66601366, 0.67198515, 0.24629530],
    [0.04680895, 0.23136024, 0.77061759],
    [0.60009728, 0.72513573, 0.06608864],
    [0.96599485, 0.86111969, 0.56682913],

    # Iteration results from our project
    [0.15183600, 0.43999000, 0.99088100],
    [0.93885800, 0.60065400, 0.07417400],
    [0.49258141, 0.61159319, 0.34017639],
    [0.64690500, 0.40797300, 0.91851900],
    [0.32844100, 0.23798100, 0.95532500],
    [0.48994600, 0.85906500, 0.51337400],
    [0.48839600, 0.62228000, 0.35109800],
    [0.53502000, 0.40623100, 0.16525300],
    [0.95980400, 0.86192900, 0.56425600],
    [0.21197300, 0.28867300, 0.34430200],
    [0.49364500, 0.85870800, 0.50807500],
    [0.59524100, 0.72529600, 0.06982600],
])

y = np.array([
    -0.11212220,
    -0.08796286,
    -0.11141465,
    -0.03483531,
    -0.04800758,
    -0.11062091,
    -0.39892551,
    -0.11386851,
    -0.13146061,
    -0.09418956,
    -0.04694741,
    -0.10596504,
    -0.11804826,
    -0.03637783,
    -0.05675837,

    # Iteration outputs from our project
    -0.393904968,
    -0.048787319,
    -0.049699456,
    -0.111301025,
    -0.27073902468816746,
    -0.042885215750335756,
    -0.03996446428663898,
    -0.10865336055983822,
    -0.07328700046470267,
    -0.060295591161217105,
    -0.033330865358791414,
    -0.05581563156993059,
])


# --------------------------------------------------
# 2. Helper functions
# --------------------------------------------------

def format_query(point, digits=6):
    """Format a query point for portal submission."""
    return "-".join(f"{value:.{digits}f}" for value in point)


def clip_to_bounds(points):
    """Keep all candidate values inside the [0, 1) domain."""
    return np.clip(points, 0.0, 0.999999)


def expected_improvement(candidates, model, y_best, xi=0.005):
    """Expected Improvement acquisition function."""
    mean, std = model.predict(candidates, return_std=True)
    std = np.maximum(std, 1e-12)

    improvement = mean - y_best - xi
    z = improvement / std

    ei = improvement * norm.cdf(z) + std * norm.pdf(z)
    return ei


def upper_confidence_bound(candidates, model, kappa=2.0):
    """Upper Confidence Bound acquisition function."""
    mean, std = model.predict(candidates, return_std=True)
    return mean + kappa * std


# --------------------------------------------------
# 3. Check data and best result
# --------------------------------------------------

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

best_index = np.argmax(y)
best_point = X[best_index]
best_y = y[best_index]

print("\nBest point so far:")
print(format_query(best_point))
print("Best output so far:", best_y)


# --------------------------------------------------
# 4. Gaussian Process model
# --------------------------------------------------
# Matern kernel is used because Function 3 is nonlinear and 3-dimensional.

kernel = (
    ConstantKernel(1.0, constant_value_bounds=(1e-3, 1e3))
    * Matern(
        length_scale=[0.2, 0.2, 0.2],
        length_scale_bounds=(1e-3, 1e3),
        nu=2.5
    )
    + WhiteKernel(
        noise_level=1e-6,
        noise_level_bounds=(1e-10, 1e-2)
    )
)

gpr = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    alpha=1e-8,
    n_restarts_optimizer=20,
    random_state=42
)

gpr.fit(X, y)

print("\nGP Model Diagnostics:")
print("Kernel:", gpr.kernel_)
print("Training score:", round(gpr.score(X, y), 4))


# --------------------------------------------------
# 5. Generate candidate points
# --------------------------------------------------
# Strategy:
# 60% local exploitation around the best point
# 40% global exploration using Latin Hypercube Sampling

np.random.seed(42)

local_candidates = best_point + np.random.normal(
    loc=0.0,
    scale=0.08,
    size=(4000, 3)
)

local_candidates = clip_to_bounds(local_candidates)

sampler = qmc.LatinHypercube(d=3, seed=42)
global_candidates = sampler.random(n=3000)

X_candidates = np.vstack([
    local_candidates,
    global_candidates
])


# --------------------------------------------------
# 6. Acquisition scoring: EI + UCB
# --------------------------------------------------

EI_XI = 0.005
UCB_KAPPA = 2.0

ei_scores = expected_improvement(
    candidates=X_candidates,
    model=gpr,
    y_best=best_y,
    xi=EI_XI
)

ucb_scores = upper_confidence_bound(
    candidates=X_candidates,
    model=gpr,
    kappa=UCB_KAPPA
)

# Normalize acquisition scores
ei_norm = (ei_scores - ei_scores.min()) / (ei_scores.max() - ei_scores.min() + 1e-12)
ucb_norm = (ucb_scores - ucb_scores.min()) / (ucb_scores.max() - ucb_scores.min() + 1e-12)

# Hybrid acquisition score
# EI focuses on improvement, UCB keeps exploration.
hybrid_score = 0.65 * ei_norm + 0.35 * ucb_norm


# --------------------------------------------------
# 7. Select next query
# --------------------------------------------------

best_candidate_index = np.argmax(hybrid_score)
x_next = X_candidates[best_candidate_index]

predicted_y, predicted_std = gpr.predict(
    x_next.reshape(1, -1),
    return_std=True
)

print("\nNext Point to Sample:")
print("X =", x_next)
print("Submission format:", format_query(x_next))
print("Predicted y:", predicted_y[0])
print("Uncertainty:", predicted_std[0])
print("Hybrid score:", hybrid_score[best_candidate_index])


# --------------------------------------------------
# 8. Show top 5 candidates
# --------------------------------------------------

top5_index = np.argsort(hybrid_score)[-5:][::-1]

print("\nTop 5 Candidates:")

for rank, idx in enumerate(top5_index, start=1):
    point = X_candidates[idx]
    mean_i, std_i = gpr.predict(point.reshape(1, -1), return_std=True)

    print(
        f"{rank}. X={format_query(point)} | "
        f"pred={mean_i[0]:.6f}, "
        f"std={std_i[0]:.6f}, "
        f"score={hybrid_score[idx]:.6f}"
    )

Shape of X: (27, 3)
Shape of y: (27,)

Best point so far:
0.493645-0.858708-0.508075
Best output so far: -0.033330865358791414

GP Model Diagnostics:
Kernel: 1.53**2 * Matern(length_scale=[1e+03, 1e+03, 0.129], nu=2.5) + WhiteKernel(noise_level=0.00867)
Training score: 0.9955

Next Point to Sample:
X = [8.21893054e-01 4.37747321e-01 2.01517197e-05]
Submission format: 0.821893-0.437747-0.000020
Predicted y: -0.031117811036380158
Uncertainty: 0.07054680980249492
Hybrid score: 0.9999999999750206

Top 5 Candidates:
1. X=0.821893-0.437747-0.000020 | pred=-0.031118, std=0.070547, score=1.000000
2. X=0.968003-0.529805-0.000346 | pred=-0.031075, std=0.070236, score=0.997097
3. X=0.408705-0.738217-0.000782 | pred=-0.031019, std=0.069820, score=0.993179
4. X=0.980409-0.290896-0.001104 | pred=-0.030978, std=0.069512, score=0.990270
5. X=0.586409-0.821986-0.001352 | pred=-0.030947, std=0.069273, score=0.988013


/usr/local/lib/python3.12/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
